In [1]:
# Install starter package
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [3]:
# Confirms that the starting package is working
# Seed is set to 42 for consistent result
times = delivery_times("Far West", "Fri/Sat eve", 45, seed=42)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.0 minutes


# Part I. Describe

## a. Percentage of late orders

In [4]:
def late_orders_pct(zone, time_block, promise):
  # Returns the percentage of orders delivered later than the promise (in minutes).
  # 'all' for zone or time_block means "include every zone / time block".

  if zone != 'all' and time_block != 'all':
    pairs = [(zone, time_block)]
  elif zone != 'all' and time_block == "all":
    pairs = [(zone, item) for item in TIME_BLOCKS]
  elif time_block != 'all' and zone == "all":
    pairs = [(item, time_block) for item in ZONES]
  else:
    pairs = [(all_zone, all_time)
               for all_zone in ZONES for all_time in TIME_BLOCKS]

  late_orders = 0
  all_orders = 0
  for z, t in pairs:
    all_times = delivery_times(z, t, promise, seed=42)
    late_orders += np.sum(all_times > promise)
    all_orders += len(all_times)

  return round(late_orders / all_orders * 100, 2)


In [7]:
# Test with a 45-minute promise: one example for each of the four cases
print("Far West, Fri/Sat eve:        ", late_orders_pct("Far West", "Fri/Sat eve", PROMISE), "%")
print("All zones, Fri/Sat eve:       ", late_orders_pct("all", "Fri/Sat eve", PROMISE), "%")
print("Far West, all time blocks:    ", late_orders_pct("Far West", "all", PROMISE), "%")
print("All zones, all time blocks:   ", late_orders_pct("all", "all", PROMISE), "%")

Far West, Fri/Sat eve:         36.67 %
All zones, Fri/Sat eve:        21.64 %
Far West, all time blocks:     15.9 %
All zones, all time blocks:    9.29 %


## b. Rank pairs of zone and time blocks in descending order of late arrival rate

In [8]:
# Late rate for every (zone, time block) pair with the current 45-minute promise
late_rates = {(z, t): late_orders_pct(z, t, PROMISE) for z in ZONES for t in TIME_BLOCKS}

# Sort late rate pairs from highest to lowest late rate
late_ranking = sorted(late_rates.items(), key=lambda pair: pair[1], reverse=True)

for rank, ((z, t), rate) in enumerate(late_ranking, start=1):
    print(f"{rank}. {z}, {t}: {rate}%")

1. Far West, Fri/Sat eve: 36.67%
2. North, Fri/Sat eve: 18.0%
3. Far West, Weekday eve: 10.59%
4. North, Weekday eve: 9.2%
5. Central, Fri/Sat eve: 7.89%
6. Central, Weekday eve: 6.97%
7. Far West, Other: 6.25%
8. North, Lunch: 4.21%
9. Central, Lunch: 3.85%
10. North, Other: 2.5%
11. Far West, Lunch: 2.5%
12. Central, Other: 2.11%


## c. Average delivery time

In [10]:
def avg_time(zone, time_block, promise):
  # Returns the average delivery time (minutes) for the selected zone(s) and time block(s).
  # 'all' for zone or time_block means "include every zone / time block".

  if zone != 'all' and time_block != 'all':
    pairs = [(zone, time_block)]
  elif zone != 'all' and time_block == "all":
    pairs = [(zone, item) for item in TIME_BLOCKS]
  elif time_block != 'all' and zone == "all":
    pairs = [(item, time_block) for item in ZONES]
  else:
    pairs = [(all_zone, all_time)
               for all_zone in ZONES for all_time in TIME_BLOCKS]

  total_minutes = 0
  all_orders = 0
  for z, t in pairs:
    all_times = delivery_times(z, t, promise, seed=42)
    total_minutes += sum(np.sum(times) for times in all_times)
    all_orders += len(all_times)

  return round(total_minutes / all_orders, 2)


In [11]:
# Test with a 45-minute promise: one example for each of the four cases
print("Far West, Fri/Sat eve:        ", avg_time("Far West", "Fri/Sat eve", PROMISE), "min")
print("All zones, Fri/Sat eve:       ", avg_time("all", "Fri/Sat eve", PROMISE), "min")
print("Far West, all time blocks:    ", avg_time("Far West", "all", PROMISE), "min")
print("All zones, all time blocks:   ", avg_time("all", "all", PROMISE), "min")

Far West, Fri/Sat eve:         42.98 min
All zones, Fri/Sat eve:        37.71 min
Far West, all time blocks:     35.38 min
All zones, all time blocks:    31.56 min


## d. Rank pairs of zone and time blocks in descending order of average delivery time

In [12]:
# Average delivery time for every (zone, time block) pair with the current 45-minute promise
avg_times = {(z, t): avg_time(z, t, PROMISE) for z in ZONES for t in TIME_BLOCKS}

# Sort average delivery time pairs from longest to shortest average delivery time
avg_ranking = sorted(avg_times.items(), key=lambda pair: pair[1], reverse=True)

for rank, ((z, t), avg) in enumerate(avg_ranking, start=1):
    print(f"{rank}. {z}, {t}: {avg} mins")

1. Far West, Fri/Sat eve: 42.98 mins
2. North, Fri/Sat eve: 37.07 mins
3. Far West, Weekday eve: 35.25 mins
4. Central, Fri/Sat eve: 32.39 mins
5. North, Weekday eve: 31.94 mins
6. Far West, Lunch: 30.37 mins
7. Far West, Other: 30.33 mins
8. Central, Weekday eve: 29.98 mins
9. North, Lunch: 28.91 mins
10. North, Other: 27.94 mins
11. Central, Lunch: 27.32 mins
12. Central, Other: 26.24 mins


## e. Comparing the relevance of late delivery rate vs average delivery time

Looking at the results from part b) and ), the late delivery rate would be more relevant to Rosa decisions.

The average delivery time is not as relevant since it doesn't show how often the 45-minute is broken. Also, from a client's perspective, if they live in the Far West and order on a Friday evening, their delivery time can be up to 50 minutes for multiple times, while on average, the delivery time is around 42-45 minutes. Thus, the average delivery time cannot reflect exactly the extent of the promise being broken. It is still, however, useful to illustrate the typical speed and overall efficiency of all deliveries.

The late delivery rate is more relevant because it measures the customer's experience of the promise. With the same example of the Far west region, on a Friday or Saturday evening: the average delivery time is in the range of high-30 to mid-40 minutes. Nevertheless, the late delivery rate is very big compared to the second high pair - North and Fri/Sat evening. Hence, the late delivery rate would be way more useful for Rosa to reconsider her resource allocation or adjust her promise.

# Part II. Exploring best promise

## a. Total cost per late order

In [13]:
# Check the key names and values in the COSTS dictionary
print(COSTS)

{'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}


In [14]:
def total_cost_per_late_order(costs):
  # Each late order costs Rosa:
  #   1. a direct refund to the customer
  #   2. lost future profit: churned orders x profit margin per order
  refund = costs['refund']
  lost_profit = costs['churn_orders'] * costs['margin']
  return round(refund + lost_profit, 2)

In [15]:
# Test with the figures in COSTS
print("Total cost per late order: $", total_cost_per_late_order(COSTS))

Total cost per late order: $ 26.2


## b. Best promised delivery time

### b. i. Choosing a range of duration to test

The duration range should start at the shortest delivery time so that almost every order would be late, and end at the longest delivery time so that almost none would be late. The best promised time will always be within this range.

Based on the last four weeks:

In [16]:
# Combine the delivery times of all pairs (45-minute promise) to find
# the shortest and longest deliveries, which set the bounds of the range to test
all_times = np.concatenate([delivery_times(z, t, PROMISE, seed=42)
                          for z in ZONES for t in TIME_BLOCKS])
shortest_time = all_times.min()
longest_time = all_times.max()
print("Shortest delivery:", shortest_time, "min")
print("Longest delivery: ", longest_time, "min")

Shortest delivery: 13.4 min
Longest delivery:  100.7 min


**Note:** If the best promised time lies at the shortest or longest delivery time, we have to widen this range. This is because it's likely we will be to find a better promised time outside this range

### b. ii. Best promise for each pair of zone and time block

The best promised time will be the time with the highest profit.

In [17]:
def net_profit(zone, time_block, promise, costs):
  # Net profit is the margin earned on all orders
  # minus the cost of the late ones
  times = delivery_times(zone, time_block, promise, seed=42)
  revenue = len(times) * costs['margin']
  late_cost = np.sum(times > promise) * total_cost_per_late_order(costs)
  return revenue - late_cost

def best_promise(zone, time_block, promises, costs):
  # Net profit for every promise in the list, stored as {promise: profit}
  # The best promise is the one with the highest net profit
  profits = {p: net_profit(zone, time_block, p, costs) for p in promises}
  best = max(profits, key=profits.get)
  return round(best, 2), round(profits[best], 2)

In [18]:
# Promises to test: from the shortest to the longest delivery time, in 5-minute steps
promises = list(np.arange(shortest_time, longest_time, 5))

# Testing for Far West region on Fri/Sat evening
best, profit = best_promise("Far West", "Fri/Sat eve", promises, COSTS)
print(f"Far West, Fri/Sat eve -> best promise: {best} min, net profit: ${profit}")

Far West, Fri/Sat eve -> best promise: 53.4 min, net profit: $1171.4


In [19]:
# Testing best promised time for all regions

for z in ZONES:
  for t in TIME_BLOCKS:
    best, profit = best_promise(z, t, promises, COSTS)
    print(f"{z}, {t}: Best promised time is: {best} mins - generating ${profit}")

Central, Lunch: Best promised time is: 48.4 mins - generating $2082.2
Central, Weekday eve: Best promised time is: 48.4 mins - generating $2510.0
Central, Fri/Sat eve: Best promised time is: 48.4 mins - generating $1418.6
Central, Other: Best promised time is: 43.4 mins - generating $1597.8
North, Lunch: Best promised time is: 48.4 mins - generating $1497.2
North, Weekday eve: Best promised time is: 48.4 mins - generating $1817.8
North, Fri/Sat eve: Best promised time is: 53.4 mins - generating $1027.6
North, Other: Best promised time is: 43.4 mins - generating $1292.6
Far West, Lunch: Best promised time is: 48.4 mins - generating $981.8
Far West, Weekday eve: Best promised time is: 53.4 mins - generating $1197.8
Far West, Fri/Sat eve: Best promised time is: 53.4 mins - generating $1171.4
Far West, Other: Best promised time is: 48.4 mins - generating $1480.0


# Part III. Creating a Web app

- GitHub Repo: https://github.com/h4hoa/Rosa-pizza-delivery
- Streamlit app link: https://rosa-pizza-delivery-hanna-nguyen.streamlit.app/